# Lab 3 Instructions

In this Lab, you will explore discrete event simulations coupled with the concepts of Packet, Packet Generator, and Packet Sink, which provides a powerful framework for modeling and analyzing network systems. By simulating packet transmission, generation, and reception, we gain valuable insights into network behavior, enabling  design and optimize network architectures, protocols, and algorithms for improved performance and efficiency.

Discrete event simulation (DES) is a powerful technique used in various fields to model complex systems characterized by discrete, sequential events occurring over time. In network simulation, DES is particularly valuable for understanding the behavior and performance of communication systems. 

## 1. Packet class
In this exercise, we will simulate packet transmission in a network using the `Packet` class from `SimComponents.py`. We have slightly modified the script so it can store and display packet information more effectively. Using **N1** traffic sources and **N2** destination nodes, we will study packet transmission times and estimate the probability of packet collisions based on a user-defined number of packets generated by each source.

To keep the simulation simple, we assume that all packets are the same size. Although this makes the analysis easier, it does not fully represent real network traffic, where packet sizes can vary and often follow a non-uniform distribution.

Recall the traffic interarrival times distribution we observed using Wireshark?


In [1]:
import random  # Import the random module for generating random numbers

# Importing the Packet class
class Packet(object):
    """ A very simple class that represents a packet.
        This packet will run through a queue at a switch output port.
        We use a float to represent the size of the packet in bytes so that
        we can compare to ideal M/M/1 queues.

        Parameters
        ----------
        time : float
            the time the packet arrives at the output queue.
        size : float
            the size of the packet in bytes
        id : int
            an identifier for the packet
        src, dst : int
            identifiers for source and destination
        flow_id : int
            small integer that can be used to identify a flow (sequence of packets between a particular source-destination pair).
            Example: flow_id=0 → first TCP flow, flow_id=1 → second TCP flow. Lets you group packets into flows for throughput, latency, or fairness analysis.
    """
    def __init__(self, time, size, id, src="a", dst="z", flow_id=0):
        # Initialize packet attributes
        self.time = time
        self.size = size
        self.id = id
        self.src = src
        self.dst = dst
        self.flow_id = flow_id

    def __repr__(self):
        # String representation of the Packet object
        return "id: {}, src: {}, dst: {}, time: {}, size: {}".format(self.id, self.src, self.dst, self.time, self.size)

# Creating packets from 4 sources to 4 destinations
sources = ["src1_IP", "src2_IP"]  # List of source IPs you can input your choice of IP addresses here
destinations = ["dst1_IP","dst2_IP"]  # List of destination IPs
packets = []  # List to store generated packets
Num_packets_src = 1000                       #int(input("number of packets generated by each source"))
# Generating packets for each source-destination pair
packet_id = 1  # Initialize packet ID
packet_sizes = [10, 100 ]   # Index 1 → 10 bytes, Index 2 → 100 bytes

# Dictionary to store the count of packets for each destination
packet_count = {dst: 0 for dst in destinations}

for src in sources:
    for k in range(Num_packets_src):
        dst_r = random.choice(destinations)  # Randomly choose destination list of destinations
        src_index = sources.index(src)
        packet = Packet(time=0, size=packet_sizes[src_index], id=packet_id, src=src, dst=dst_r)  # Create Packet object
        packet_count[dst_r] += 1  # Increment packet count for the chosen destination
        packets.append(packet)  # Add packet to the list
        packet_id += 1  # Increment packet ID

# Updating time for each packet with random time increments
for packet in packets:
    random_time_increment = random.uniform(0.1, 1)  # Generate random time increment between 0.1 and 1
    packet.time = random_time_increment  # Update packet arrival time
# Printing the generated packets with updated time
for packet in packets:
    print(packet)  # Print packet details
# Printing the number of packets generated for each destination
for dst, count in packet_count.items():
    print("Destination {}: {} packets generated".format(dst, count))
    


id: 1, src: src1_IP, dst: dst1_IP, time: 0.9737117530181825, size: 10
id: 2, src: src1_IP, dst: dst1_IP, time: 0.975459620596205, size: 10
id: 3, src: src1_IP, dst: dst1_IP, time: 0.8415662402453434, size: 10
id: 4, src: src1_IP, dst: dst2_IP, time: 0.8023028607776304, size: 10
id: 5, src: src1_IP, dst: dst2_IP, time: 0.7892242236709163, size: 10
id: 6, src: src1_IP, dst: dst2_IP, time: 0.562476475189367, size: 10
id: 7, src: src1_IP, dst: dst1_IP, time: 0.16738913872731498, size: 10
id: 8, src: src1_IP, dst: dst2_IP, time: 0.4712398212873754, size: 10
id: 9, src: src1_IP, dst: dst1_IP, time: 0.7475171480953937, size: 10
id: 10, src: src1_IP, dst: dst1_IP, time: 0.7923686132097535, size: 10
id: 11, src: src1_IP, dst: dst2_IP, time: 0.1491547458924203, size: 10
id: 12, src: src1_IP, dst: dst2_IP, time: 0.10131115238170356, size: 10
id: 13, src: src1_IP, dst: dst2_IP, time: 0.5103803628975745, size: 10
id: 14, src: src1_IP, dst: dst1_IP, time: 0.7865144004356421, size: 10
id: 15, src: sr

In [2]:
from collections import defaultdict
import numpy as np

# Collect arrival times per destination
times_by_dst = defaultdict(list)
for p in packets:
    times_by_dst[p.dst].append(p.time)

# Max times per destination (handles empty lists safely)
max_time_dst1 = max(times_by_dst.get("dst1_IP", []), default=None)  #use the maximum of times_by_dst.get("dst1_IP", []), default=None) to get maximum time dst1 is receiving packets
max_time_dst2 = max(times_by_dst.get("dst2_IP", []), default=None)
min_time_dst1 = min(times_by_dst.get("dst1_IP", []), default=None)
min_time_dst2 = min(times_by_dst.get("dst2_IP", []), default=None)

print("Maximum time for dst1_IP:", max_time_dst1)
print("Maximum time for dst2_IP:", max_time_dst2)
print("Minimum time for dst1_IP:", min_time_dst1)
print("Minimum time for dst2_IP:", min_time_dst2)

# Arrival rate per destination = count / (max time for that destination)
for dst, count in packet_count.items():
    dst_max_time = max(times_by_dst.get(dst, []), default=None)
    #Write the code to complete to calculate packet arrival rates of dst1 and dst2.
    if dst_max_time is not None and dst_max_time > 0:
        arrival_rate = count / dst_max_time
        print("Arrival rate for {}: {:.2f} packets/sec".format(dst, arrival_rate))
    else:
        print("No packets or invalid max time for {}".format(dst))
     

Maximum time for dst1_IP: 0.9973174165092255
Maximum time for dst2_IP: 0.9999515186119556
Minimum time for dst1_IP: 0.1012198000108053
Minimum time for dst2_IP: 0.10009023075769785
Arrival rate for dst1_IP: 978.63 packets/sec
Arrival rate for dst2_IP: 1024.05 packets/sec


### Exercise 1
    
1. Complete the code with the help of the comments. Choose the number of packets per information source.  INPUT 

2. In the simulation, the *time* parameter is utilized to denote the arrival time of packets at the output queue within the network. This time value determines when each packet arrives at its destination. The duration of this time slot is contingent upon the random time increments added to the initial time of each packet. What are the minimum and maximum time values  generated during the simulation?  INCLUDE CODE 

Maximum time for dst1_IP: 0.9999436874722286  
Maximum time for dst2_IP: 0.9993614656173055  
Minimum time for dst1_IP: 0.10150910333183379  
Minimum time for dst2_IP: 0.10243254698638933  

4. Calculate the packet arrival rate from each source. INCLUDE CODE 

Arrival rate for dst1_IP: 1012.06 packets/sec  
Arrival rate for dst2_IP: 988.63 packets/sec

3. Within the lab context, the Packet class serves to represent individual packets within a network simulation. It encapsulates various attributes such as arrival time, size, source, destination, and flow ID, allowing for the modeling and characterization of packets within the simulated environment. Does the Packet class specify the generation of packets as a sequential process over time?

No. It just store data about the packet.

4. As this execise involves the generation of packets on an individual basis, certain conditions must be met to ensure that two packets do not collide. What are the important considerations to check if the packets have a chance of collision in the network. Think of packet generation times, packet durations, and IP addresses.

Packet generation times: if two packets destined to the same transmission resource arrive close enough in time that their transmissions overlap, they can collide.
Packet durations: does this mean transmission duration? If so, then larger packets or lower bitrates increase the transmission time. Higher transmission time means that a packet is existing on a link for longer, so there is a higher chance that another packet comes along that wants to use that link, hence a collision.
IP addresses: packets sent to the same destination or same output queue are primary candidates for collision, as they are likely to try exist on the same link at the same time.

## 2. Packet Generator 

While the Packet class itself does not inherently specify the generation of packets as a sequential process over time, it can be integrated into a simulation framework such as SimPy to facilitate sequential packet generation; this is the core idea of discrete-event simulations. In this exercise we will simulate packet generation from two sources and connect it to two desitation nodes. Examine the code below for Packet Generator class.  In Packet Class, Packets are generated individually and explicitly using the Packet class constructor. Each packet is created with specific attributes such as arrival time, size, source, destination, and flow ID. However, in 
Packet Generator Class, Packets are generated in a continuous manner by an instance of the PacketGenerator class. The generator utilizes a specified inter-arrival time distribution (adist) and packet size distribution (sdist) to generate packets over time. Packets are automatically generated based on these distributions.

### Exercise 2.1

The packet generator Class of SimComponents.py uses the below code to simulate generation of packets over time.


    
    # Delay the start of the process by the initial delay time
            yield self.env.timeout(self.initial_delay)
    # Continue the process until the current time is less than the finish time
    while self.env.now < self.finish:
    # Wait for the next transmission time, determined by the adist() function
    yield self.env.timeout(self.adist())
    
    # Increment the counter for packets sent
    self.packets_sent += 1
    
    # Create a new packet at the current time with a size determined by sdist(),
    # a unique identifier, the source ID, and the flow ID
    p = Packet(self.env.now, self.sdist(), self.packets_sent, src=self.id, flow_id=self.flow_id)
    
    # Place the newly created packet in the output queue
    self.out.put(p)

            

1. Examine the code snippet from the Packet Generator class in SimComponents.py  and explain how the packets are generated.

It takes as input:

env: the SimPy environment.  
adist: a function that returns the next inter-arrival time (how long to wait before generating the next packet).  
sdist: a function that returns the size of the next packet.  
initial_delay: how long to wait before starting packet generation.  
finish: the simulation time at which to stop generating packets.  

In the run method (which is started as a SimPy process):  
It waits for the initial_delay.  
Then, in a loop until the simulation time reaches finish:  
Waits for the next inter-arrival time (yield self.env.timeout(self.adist())).  
Increments the packet count.  
Creates a new Packet object with the current time, a size from sdist(), and other metadata.  
Sends the packet to the next component (self.out.put(p)).  
  
Summary: Packets are generated at random intervals determined by adist(), with sizes determined by sdist(), and are sent to the next simulation component. The process continues until the specified finish time.



2. What are the additional attributes to the generated packet compared to that generated by the Packet Class?
3. Print interarrival times and packet sizes and recommend a way to model memorylessness in the packet generation.
4. Examine the code below and comment on whether this packet generator demonstrates the memoryless property. Write the code to check whether the interarrival times satisfy memorylessness. Recall the definition of the memoryless property for uniform, exponential, and Gaussian distributions. (Check the Workshop 3 Python resource for this.)

                                      

In [14]:
import numpy as np
import simpy
import matplotlib.pyplot as plt
from SimComponents import PacketGenerator, PacketSink

# Global lists to store packet inter-arrival times and sizes
pk_gen1_interarrival_times = []
pk_gen2_interarrival_times = []
pk_gen1_packet_sizes = []
pk_gen2_packet_sizes = []

def arrival_dist1():
    # Constant arrival distribution for generator 1
    interarrival_time = 
    return interarrival_time

def arrival_dist2():
    # Constant arrival distribution for generator 2
    interarrival_time = 
    return interarrival_time

def size_dist1():
    # Constant  packet sizes in bytes for generators
    size = 
    return size
def size_dist2():
    # Constant packet sizes in bytes for generators
    size = 
    return size

def two_pk_generators():
    # Create the SimPy environment
    env = simpy.Environment()
    pk_sink = PacketSink(env, debug=True)  # Enable debugging for simple output
    pk_gen1 = PacketGenerator(env, "flow_1", arrival_dist1, size_dist1)
    pk_gen2 = PacketGenerator(env, "flow_2", arrival_dist2, size_dist2)

    # Connect packet generators to the sink
    pk_gen1.out = pk_sink
    pk_gen2.out = pk_sink
    
    env.run(until=10)

if __name__ == '__main__':
    two_pk_generators()

# Confirm if the packet inter arrival times satisfy memorylessness property

# INCLUDE CODE HERE







SyntaxError: invalid syntax (1668776147.py, line 14)

### Exercise 2.2

In this exercise you will extend the code to include the property of memorylessness in the packet generation process.

Memorylessness is a fundamental concept in probability theory and stochastic processes that simplifies mathematical modeling and facilitates pcyanictions of future events. It plays a crucial role in network performance analysis and various other fields. 

Memorylessness is often represented using exponential distributions in continuous-time processes and geometric distributions in discrete-time processes. These distributions have simple forms and are well-understood, making them valuable in a wide range of applications.

In this exercise, we will generate packets from two packet generators. These generators produce packets with sizes and inter-arrival times drawn from exponential distributions. Through this simulation, we aim to demonstrate the memorylessness of tiarrival times and packet sizes can be used in for network modelling performance analysis.

1. Enter arrival rate for Pkt Gen 1, arrival rate for Pkt Gen 2, Packet size in bytes for Gen1 and Packet size in bytes for Gen2. Enter the Simulation time =100. complete the code to choose interarrival times drawn from exponential distribution. 
2. Run the code below and examine the output displayed,  write the first 3 times for Gen1 and Gen2 based on their time stamps flow_ids. What do you observe in terms of their time between events for Gen1 and Gen2? Is the latency same for both generators? How is this different ot Packet Class in the previosu exercise?
3. Now look at the packet sizes generated Gen1 and Gen2. Write the code to calculate the mean of the packet sizes (pk_gen1_packet_sizes) (pk_gen2_packet_sizes). Is this expected?






In [ ]:
import numpy as np
import simpy
import matplotlib.pyplot as plt
from SimComponents import PacketGenerator, PacketSink

# Global lists to store packet inter-arrival times and sizes
pk_gen1_interarrival_times = []
pk_gen2_interarrival_times = []
pk_gen1_packet_sizes = []
pk_gen2_packet_sizes = []

def arrival_dist1():
    # Constant arrival distribution for generator 1
    a1 =             #Enter arrival rate for Pkt Gen 1
    interarrival_time =              # random inter arrival times drawn from the exponential distribution with a rate parameter a1
    pk_gen1_interarrival_times.append(interarrival_time)
    return interarrival_time

def arrival_dist2():
    # Constant arrival distribution for generator 2
    a2 =  #Enter arrival rate for Pkt Gen 2
    interarrival_time =            #random inter arrival times drawn from the exponential distribution with a rate parameter a2
    pk_gen2_interarrival_times.append(interarrival_time)
    return interarrival_time

def size_dist1():
    # Exponential distribution of packet sizes for generators
    sz1 =       #Enter average Pkt Size of Gen1
    size =            #random exponential distribution with a rate parameter a2
    pk_gen1_packet_sizes.append(size)
    return size
def size_dist2():
    # Exponential distribution of packet sizes for generators
    sz2 =     #Enter average Pkt Size of Gen2
    size =     
    pk_gen2_packet_sizes.append(size)
    return size

def two_pk_generators():
    # Create the SimPy environment
    env = simpy.Environment()
    pk_sink = PacketSink(env, debug=False)  # Enable debugging for simple output
    pk_gen1 = PacketGenerator(env, "Gen1", arrival_dist1, size_dist1)
    pk_gen2 = PacketGenerator(env, "Gen2", arrival_dist2, size_dist2)

    # Connect packet generators to the sink
    pk_gen1.out = pk_sink
    pk_gen2.out = pk_sink
    
    
    SimTime = input("Enter the Simulation time: ")
    env.run(until=SimTime)

   

if __name__ == '__main__':
    two_pk_generators()


In [ ]:
# write the code to calculate the mean of the packet sizes (pk_gen1_packet_sizes) (pk_gen2_packet_sizes), 
#  use np.mean or write your own from first principles



Mean_Inter_arrival_Gen1 = 
Mean_Inter_arrival_Gen2 = 




4. Turn the debug=False in PacketSink call to disable printing packet information.  Now, run the code again and input a large number for simulatiopn time and then plot histograms of inter-arrival times and packet sizes

In [ ]:
#### # Plot histograms of packet inter-arrival times and sizes for pk_gen1
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)

plt.xlabel('Inter-arrival Time')
plt.ylabel('Probability Density')
plt.title('Histogram of pk_gen1 Inter-arrival Times')
plt.grid(True)
    
plt.subplot(1, 2, 2)

plt.xlabel('Packet Size')
plt.ylabel('Probability Density')
plt.title('Histogram of pk_gen1 Packet Sizes')
plt.grid(True)
    
plt.tight_layout()
plt.show()

    # Plot histograms of packet inter-arrival times and sizes for pk_gen2
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)

plt.xlabel('Inter-arrival Time')
plt.ylabel('Probability Density')
plt.title('Histogram of pk_gen2 Inter-arrival Times')
plt.grid(True)
    
plt.subplot(1, 2, 2)

plt.xlabel('Packet Size')
plt.ylabel('Probability Density')
plt.title('Histogram of pk_gen2 Packet Sizes')
plt.grid(True)
    
plt.tight_layout()
plt.show()


5. From the plots what can be said about interarrival times and sizes on a network?  Is this what you would expect in a real scenario?

### Exercise 2.3
 Number of arrivals given exponential inter arrivals. Use the mean interarrival time you calculated above to check the number of arrivals within a certain time limit, say 1 second. Plot the histogram of many such trials. That is we will use this as the time average `Mean_Inter_arrival_Gen1` to simulate multiple trials and calculate the number of arrivals within a small time slot. Then, plot the histogram of the number of arrivals and observe its distribution.
 1. What is it
 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def inter_arrival_times(rate, time_limit):
    """
    Simulate a Poisson process given a rate (λ) and time limit.

    Parameters:
    rate (float): The rate parameter λ of the exponential distribution.
    time_limit (float): The total time duration to simulate.

    Returns:
    event_times (list): The times at which events occur (t_1, t_2, ... ≤ time_limit).
    interarrival_times (list): The interarrival times actually used (Δ_1, Δ_2, ...).
    """
    event_times = []
    interarrival_times = []
    current_time = 0.0

    while True:
        inter = np.random.exponential(1.0 / rate)
        if current_time + inter > time_limit:
            break  # do not include the overshoot interval
        current_time += inter
        event_times.append(current_time)
        interarrival_times.append(inter)

    return event_times, interarrival_times

def simulate_multiple_processes(rate, time_limit, num_simulations):

    num_events_list = []
    for _ in range(num_simulations):
        event_times, _ = inter_arrival_times(rate, time_limit)
        num_events_list.append(len(event_times))
    return num_events_list





# Parameters
TI = Mean_Inter_arrival_Gen1
rate = 1.0 / TI  # Rate λ
time_limit = 0.1
num_simulations = 1000

# Simulate one process (for plots 1 & 2) and many (for plot 3)
one_event_times, one_interarrival_times = inter_arrival_times(rate, time_limit)
num_events_list = simulate_multiple_processes(rate, time_limit, num_simulations)

# Plotting
plt.figure(figsize=(18, 6))

# 1) Histogram of interarrival times from one simulated process
plt.subplot(1, 2, 1)

#     plt.hist( )
plt.title('Histogram of Interarrival Times (one run)')
plt.xlabel('Interarrival Time')
plt.ylabel('Frequency')



# 3) Histogram of number of events across simulations
plt.subplot(1, 2, 2)

# plt.hist( )
plt.title('Histogram of Number of Events per Simulation')
plt.xlabel('Number of Events')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

### Exercise 2.4

    
1. Calculate the mean and variance of the number of events (arrivals), and predict what the distribution could be.
2. Check the mean and variance of the multiple trials stored in `num_events_list`.
3. What is the connection between interarrival times and the number of arrivals?
   


 


### Calculating CDF in Python
Examine the code below and answer the following


    
4. Complete the code below.

5. What information is conveyed in the plot of cumulative distribution function (CDF)?
- [Recall, Mathematically, the CDF of a random variable $ X $, denoted as $ F(x) $, is defined as:
  $$ F(x) = P(X \leq x) $$
  In other words, the CDF gives the cumulative probability of observing a value less than or equal to $ x $ from the probability   distribution of $ X $. It provides a complete summary of the distribution of $ X $.]

If you have a dataset of size \(N\), and the sorted values are  
$$
x_{(1)} \leq x_{(2)} \leq \dots \leq x_{(N)},
$$
then the empirical CDF at the \(i\)-th sorted point is  
$$
F(x_{(i)}) = \frac{i}{N}, \quad i = 1, 2, \dots, N
$$
In NumPy, this is implemented as:
cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)





In [ ]:
# Calculate CDF for inter-arrival times
gen1_interarrival_times = np.array(pk_gen1_interarrival_times)
gen2_interarrival_times = np.array(pk_gen2_interarrival_times)

                            #sort the data gen1_interarrival_times 
                                #sort the data gen2_interarrival_times 

gen1_cdf =                     # Calculate cdf
gen2_cdf =                      # Calculate cdf

# Plotting
plt.plot(gen1_interarrival_times, gen1_cdf, label='Gen1', marker='o')
plt.plot(gen2_interarrival_times, gen2_cdf, label='Gen2', marker='o')

plt.xlabel('Inter-arrival Time')
plt.ylabel('Probability')
plt.title('CDF of Inter-arrival Times for Generators')
plt.legend()
plt.grid(True)
plt.show()


6. Complete the code below to calculate  the probability of inter-arrival time less than 0.5secs.



In [ ]:
# Find the index where inter-arrival time is just greater than arrival_time
arrival_time = float(input("inter arrival time threshold: " ))
index1 =     #   index of the first element in gen2_interarrival_times that is greater than arrival_time. np.argmax(gen1_interarrival_times > arrival_time )
index2 =     #   index of the first element in gen2_interarrival_times that is greater than arrival_time. np.argmax(gen1_interarrival_times > arrival_time )
# Calculate the probability using the CDF array
probability1 = gen1_cdf[index1-1]
probability2 = gen2_cdf[index2-1]
print("Probability inter-arrival time is just greater than arrival_time  for Gen1:", probability1)
print("Probability inter-arrival time is just greater than arrival_time  for Gen2:", probability2)

## 3. Packet Sink

PacketSink class in SimComponents.py simulates a system receiving packets and collecting delay information. In this exercise we will examine attributes of this class for use in future applications of Network modelling and analysis. 

1. Examine the python code for the PacketSink class and list out the attributes. What is the absolute_arrivals attribute used 

   for, and what instances are allowed for this attribute?

   def put(self, pkt): of this class is where the attributes are used in recording information about packet arrivals. In short 
   the code is performing the below operations. 

   The `put` method of the `PacketSink` class is designed to handle incoming packets. It first checks if there's a selector 
   function defined and if it returns `True` for the current packet (`pkt`). If there's no selector or the selector returns 
   `True`, it proceeds to process the packet. It records the current simulation time (`now`) and calculates the waiting time 
   experienced by the packet if `rec_waits` is enabled. Then, depending on whether `rec_arrivals` is set, it records the 
   arrival time of the packet, either as an absolute time if `absolute_arrivals` is enabled, or as the time difference from the 
   last arrival if not. It updates the counters for received packets and bytes, and if debugging is enabled, it prints 
   information about the packet.
    
2. In the packet generation code below we simulate a simple packet generator connected  to a packet sink.  
   pk_sink = PacketSink(env, debug=False)  # Enable debugging for simple output debug=True
   Based on your understanding of other attributes avialable, rewrite pk_sink so that it records the time 
   difference from the  last arrival.
    
3. Choose a packet size so that the transmission rate is no more than 10Mbps. What is the minimum value of packet size   
   that satisfies this constrain?


In [ ]:
from random import expovariate
import numpy as np
import simpy
import matplotlib.pyplot as plt

from SimComponents import PacketGenerator, PacketSink, SwitchPort

def arrival_dist():
    # exponential arrival distribution for generator 2
    a2 =  
    interarrival_time = 
    
    return interarrival_time

def size_dist():
    # Exponential distribution of packet sizes for generators
    sz1 = 
    size = 
  
    return size

env = simpy.Environment()  # Create the SimPy environment
ps = PacketSink(env, debug=False) # debug: every packet arrival is printed
pg = PacketGenerator(env, "Pkt_Sink", arrival_dist, size_dist)

# Wire packet generators and sinks together
pg.out =  ps
env.run(until=100)
print("waits: {}".format(sum(ps.waits)))
print("received: {}, sent {}".format(ps.packets_rec,
 pg.packets_sent))

# Calculate average wait time
total_wait_time = sum(ps.waits)
average_wait_time = total_wait_time / len(ps.waits)
average_arrival_rate =  size_dist() *8 /arrival_dist()
print("Average wait time: {:.2f} time units".format(average_wait_time))
print("Average arrival rate: {:.2f} bits per second".format(average_arrival_rate))


 In this lab we explored the classes for packet generation, packet handling, and packet sink functionalities, which collectively simulate a network environment. By analyzing these classes and their interactions, several aspects of network performance can be examined. Specifically, explain the following:

1. **Packet Generation Rate**  
   - How is the rate of packet generation determined in the simulation?  
   - What parameters or random distributions control how frequently packets are generated?  
   

2. **Packet Handling and Queuing**  
    - What queuing discipline (e.g., FIFO, priority-based) is implemented in the packet handler?  
  

3. **Traffic Patterns and Variability**  
   - What traffic arrival patterns are modeled?  (Think of what the exponential distribution of inter arrival times say about the traffic? 
   - How does variability in inter-arrival times or packet sizes impact overall network performance?  
   - Can the simulation capture real-world traffic irregularities?


